In [1]:
import os
from dotenv import load_dotenv
from src.model_compare import compare_models
from src.chat import create_client, chat_turn
import pandas as pd

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
client = create_client(api_key)
print("Client ready.")

Client ready.


In [5]:
MODELS = [
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    #"openai/gpt-oss-120b:free",
    "poolside/laguna-m.1:free",
]

PROMPT = "Explain what a neural network is in 3 sentences, like I am 16 years old."

results = compare_models(client, MODELS, PROMPT)

for r in results:
    print(f"\n--- {r['model']} ({r['time']:.2f}s) ---")
    print(r['response'])

Querying nvidia/nemotron-3-ultra-550b-a55b:free...
Querying poolside/laguna-m.1:free...

--- nvidia/nemotron-3-ultra-550b-a55b:free (14.48s) ---
A neural network is a computer system modeled loosely on the human brain, made of layers of interconnected "neurons" that pass data to one another. It learns by analyzing massive amounts of examples—like photos or text—and automatically adjusting the strength of its internal connections to minimize errors. Once trained, it can recognize complex patterns in new data it has never seen before, powering things like facial recognition, language translation, and game-playing AIs.

--- poolside/laguna-m.1:free (13.69s) ---

A neural network is like a computer program designed to mimic how the brain works, using layers of connected "neurons" that pass information along to make decisions or predictions. Each connection has a "weight" that adjusts as the network learns from examples, helping it recognize patterns—like identifying faces in photos or pred

In [6]:
df = pd.DataFrame([
    {
        "Model": r["model"],
        "Response Time (s)": round(r["time"], 2),
        "Response Preview": r["response"][:120] + "..."
    }
    for r in results
])
df

,Model,Response Time (s),Response Preview
0,nvidia/nemotron-3-ultra-550b-a55b:free,14.48,A neural network is a computer system modeled ...
1,poolside/laguna-m.1:free,13.69,\nA neural network is like a computer program ...


In [8]:
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
history = []

reply, history = chat_turn(client, MODEL, history, "What is a transformer in machine learning?")
print(f"Assistant: {reply}\n")

reply, history = chat_turn(client, MODEL, history, "How is it different from an RNN?")
print(f"Assistant: {reply}\n")

reply, history = chat_turn(client, MODEL, history, "Can you give me a real world example of where transformers are used?")
print(f"Assistant: {reply}\n")

Assistant: A **Transformer** is a deep learning architecture introduced in the seminal 2017 paper **"Attention Is All You Need"** by Vaswani et al. (Google Brain). It has become the **dominant architecture for Natural Language Processing (NLP)** and is increasingly used in Computer Vision, Audio, and Multimodal tasks.

It was designed to solve the limitations of Recurrent Neural Networks (RNNs/LSTMs) and Convolutional Neural Networks (CNNs) for sequence modeling.

---

### 1. The Core Innovation: **Self-Attention**
The defining component of the Transformer is the **Self-Attention Mechanism** (specifically *Scaled Dot-Product Attention*).

*   **The Problem with RNNs:** RNNs process tokens sequentially (word 1, then word 2, etc.). This prevents parallelization during training and makes it hard to learn relationships between distant words (vanishing gradient problem).
*   **The Transformer Solution:** Self-attention allows the model to **look at all tokens in a sequence simultaneously** 

## Week 2 Findings

### Model Comparison
Three models were compared on the same prompt via OpenRouter API.
Response time and quality were recorded in the table above.
- mistral-7b is fastest and cheapest, good for high volume tasks
- llama-3-8b gives more detailed responses, good balance of quality and cost
- claude-3-haiku gives the most natural responses, best quality among the three

### Multi-Turn Chat
The chat function in src/chat.py maintains full conversation history
by appending each message to a list and sending the entire list on
every API call. This is how multi-turn conversation is simulated
since LLMs are stateless between calls. The third question above
references the previous answers proving memory works correctly.

### Local Models
llama3 and mistral were also run locally via Ollama. Local models
have zero API cost and full privacy but are slower and less capable
than hosted models like Claude.

### Key Concepts Learned
- Tokens: LLMs read chunks of text not individual words
- Temperature: controls randomness of output, 0 is deterministic, 1 is creative
- Context window: how much text the model can see at once
- System prompt: hidden instruction given to the model before conversation starts